In [ ]:
# ── CELL 1: System & Python Dependencies ─────────────────────────────────────
# Run time: ~2 minutes. Run once per session.

# Install Tesseract OCR at system level
!apt-get install -y tesseract-ocr tesseract-ocr-eng libgl1 > /dev/null 2>&1
print("Tesseract installed")

# Install all Python packages
!pip install -q \
    watchdog \
    pytesseract \
    opencv-python-headless \
    pillow \
    spacy \
    chromadb \
    sentence-transformers \
    groq \
    PyMuPDF \
    pandas \
    python-dotenv \
    langdetect

print("Python packages installed")

# Download spaCy English model (for PII detection)
!python -m spacy download en_core_web_sm -q
print("spaCy model downloaded")

# Create folder structure
import os
for folder in ['inbox', 'processed', 'knowledge_base', 'exports']:
    os.makedirs(folder, exist_ok=True)
print("Folders created: inbox/ processed/ knowledge_base/ exports/")

print("ClawSight environment ready. Run Cell 2 next.")

Tesseract installed
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 47.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.1 

In [ ]:
# ── CELL 2: LLM Setup (Groq — updated model) ─────────────────────────────────
import os
from groq import Groq

# PASTE YOUR GROQ KEY HERE
GROQ_API_KEY = "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"

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
client = Groq(api_key=GROQ_API_KEY)

# ── Updated model name ────────────────────────────────────────────────────────
# llama3-8b-8192 was decommissioned. Use llama-3.3-70b-versatile instead.
# Still completely free on Groq's free tier.
MODEL = "llama-3.3-70b-versatile"

# Quick sanity check
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with just the word: READY"}],
    max_tokens=10,
)
print(f"Llama connected. Model says: '{test.choices[0].message.content.strip()}'")
print(f"   Model in use: {MODEL}")

Llama connected. Model says: 'READY'
   Model in use: llama-3.3-70b-versatile


In [ ]:
# ── CELL 3: File Watcher ──────────────────────────────────────────────────────
# This runs a background thread that watches inbox/ for new images.
# When you upload a file to inbox/, it automatically queues it for processing.

import time
import queue
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# Shared queue between watcher and pipeline
image_queue = queue.Queue()
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

class ClawSightWatcher(FileSystemEventHandler):
    def on_created(self, event):
        if not event.is_directory:
            ext = os.path.splitext(event.src_path)[1].lower()
            if ext in SUPPORTED_EXTENSIONS:
                print(f"\n📁 New image detected: {event.src_path}")
                image_queue.put(event.src_path)

# Start the observer
observer = Observer()
observer.schedule(ClawSightWatcher(), path='inbox/', recursive=False)
observer.start()

print("👁️  Watcher is LIVE — monitoring inbox/")
print("   To trigger: use the 📁 Files panel (left sidebar) → upload to inbox/")
print("   Or run:  !cp your_image.jpg inbox/")
print("\n   Watcher runs in the background. Continue to Cell 4.")

👁️  Watcher is LIVE — monitoring inbox/
   To trigger: use the 📁 Files panel (left sidebar) → upload to inbox/
   Or run:  !cp your_image.jpg inbox/

   Watcher runs in the background. Continue to Cell 4.


In [ ]:
# ── CELL 4: OCR with Image Preprocessing ─────────────────────────────────────
# OpenCV preprocessing before Tesseract dramatically improves accuracy.
# Handles: blurry photos, uneven lighting, small text, compressed screenshots.

import cv2
import pytesseract
import numpy as np
from PIL import Image

def preprocess_image(image_path):
    """
    Clean up a real-world phone image before OCR.
    Returns a processed numpy array ready for Tesseract.
    """
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Step 1: Upscale if image is small (phone thumbnails, compressed)
    h, w = img.shape[:2]
    if w < 1000:
        scale = 1500 / w
        img = cv2.resize(img, None, fx=scale, fy=scale,
                         interpolation=cv2.INTER_CUBIC)
        print(f"   Upscaled: {w}px → {int(w*scale)}px")

    # Step 2: Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 3: Denoise — removes JPEG compression noise
    denoised = cv2.fastNlMeansDenoising(gray, h=10)

    # Step 4: Adaptive thresholding — handles shadows and uneven lighting
    thresh = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=11,
        C=2
    )
    return thresh

def run_ocr(image_path):
    """
    Run OCR on an image. Returns (text, confidence_score).
    Tries PSM 6 first (document layout), falls back to PSM 11 (sparse/mixed).
    """
    print(f"\n🔍 Running OCR on: {image_path}")
    processed = preprocess_image(image_path)

    # PSM 6 = uniform block of text (best for slides, documents, receipts)
    # PSM 11 = sparse text (best for screenshots, mixed layouts)
    # OEM 3 = use LSTM neural net engine (most accurate)
    raw_text = pytesseract.image_to_string(
        processed,
        config='--psm 6 --oem 3'
    )

    # Get confidence scores per word
    data = pytesseract.image_to_data(
        processed,
        output_type=pytesseract.Output.DATAFRAME
    )
    valid = data[data.conf > 0]['conf']
    avg_conf = valid.mean() if len(valid) > 0 else 0.0

    # If PSM 6 gives low confidence, retry with PSM 11
    if avg_conf < 50 and len(raw_text.strip()) < 20:
        print(f"   PSM 6 confidence low ({avg_conf:.1f}%), retrying with PSM 11...")
        raw_text = pytesseract.image_to_string(
            processed,
            config='--psm 11 --oem 3'
        )
        data = pytesseract.image_to_data(
            processed,
            output_type=pytesseract.Output.DATAFRAME
        )
        valid = data[data.conf > 0]['conf']
        avg_conf = valid.mean() if len(valid) > 0 else 0.0

    text = raw_text.strip()
    print(f"   ✅ OCR done — {len(text)} chars extracted, confidence: {avg_conf:.1f}%")

    if avg_conf < 40:
        print("   ⚠️  Low confidence. Check image quality (blur, dark lighting).")

    return text, avg_conf

print("✅ OCR functions loaded. Ready for Cell 5.")

✅ OCR functions loaded. Ready for Cell 5.


In [ ]:
# ── CELL 5: PII Redaction ─────────────────────────────────────────────────────
# This is the HARD GATE. Runs before any LLM call.
# Two passes: Regex (fast, structured) + spaCy NER (catches names and orgs).
# If redaction throws any error, returns None and pipeline halts for that image.

import re
import spacy

print("Loading spaCy NER model...")
nlp = spacy.load('en_core_web_sm')
print("✅ spaCy loaded")

# ── Regex patterns ────────────────────────────────────────────────────────────
# Covers structured Indian + international PII
PII_PATTERNS = {
    'email':    r'[\w.+\-]+@[\w\-]+\.[\w.\-]+',
    'phone_in': r'(\+91[\-\s]?)?[6-9]\d{9}',           # Indian mobile
    'phone_int':r'\+?1?\s?\(?\d{3}\)?[\s.\-]\d{3}[\s.\-]\d{4}',  # International
    'aadhaar':  r'\b\d{4}\s\d{4}\s\d{4}\b',
    'pan':      r'\b[A-Z]{5}[0-9]{4}[A-Z]\b',
    'password': r'(?i)(password|passwd|pwd)\s*[:=]\s*\S+',
    'api_key':  r'(?i)(api[_\-]?key|token|secret|bearer)\s*[:=]\s*[A-Za-z0-9_\-]{20,}',
    'credit_card': r'\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b',
    'ip_address': r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b',
}

def redact_text(text):
    """
    Pass 1: Regex — catches structured PII (fast, deterministic)
    Pass 2: spaCy NER — catches names, orgs, locations (flexible)
    """
    redaction_log = []

    # Pass 1: Regex
    for pii_type, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            redaction_log.append(f"{pii_type}: {len(matches)} instance(s)")
        text = re.sub(pattern, f'[REDACTED_{pii_type.upper()}]', text)

    # Pass 2: spaCy NER (process in chunks if text is long)
    chunk_size = 100000  # spaCy default limit
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    result_parts = []

    for chunk in chunks:
        doc = nlp(chunk)
        # Reverse iteration preserves character positions during replacement
        for ent in reversed(doc.ents):
            if ent.label_ in {'PERSON', 'ORG', 'GPE', 'LOC', 'NORP'}:
                redaction_log.append(f"NER {ent.label_}: '{ent.text}'")
                chunk = chunk[:ent.start_char] + f'[REDACTED_{ent.label_}]' + chunk[ent.end_char:]
        result_parts.append(chunk)

    final = ''.join(result_parts)

    if redaction_log:
        print(f"   🔒 Redacted: {', '.join(redaction_log[:5])}" +
              (f" (+{len(redaction_log)-5} more)" if len(redaction_log) > 5 else ""))
    else:
        print("   🔒 No PII detected")

    return final

def safe_redact(raw_text):
    """Hard gate wrapper. Returns None on any failure — halts pipeline."""
    try:
        redacted = redact_text(raw_text)
        return redacted
    except Exception as e:
        print(f"   🚨 REDACTION FAILED: {e}")
        print("   Pipeline halted for this image. No data sent to LLM.")
        return None

print("✅ Redaction functions loaded. Ready for Cell 6.")

Loading spaCy NER model...
✅ spaCy loaded
✅ Redaction functions loaded. Ready for Cell 6.


In [ ]:
# ── CELL 6: Intent Classification via Groq + LLaMA 3 ─────────────────────────
# Sends redacted text to LLaMA 3 8B (free via Groq).
# Returns structured JSON: type, confidence, summary, key_data, tags.

import json

SYSTEM_PROMPT = """You are ClawSight's intent classification engine.

Given OCR text extracted from a phone image, classify it and return ONLY a valid JSON object.

The JSON must have exactly these fields:
{
  "type": "<one of: ticket | research | bill | code_error | social_rec | handwritten | other>",
  "confidence": <float between 0.0 and 1.0>,
  "summary": "<one sentence describing what this image contains>",
  "key_data": {<extracted fields relevant to the type, see below>},
  "tags": [<2 to 5 topic tags as strings>]
}

key_data extraction rules by type:
- ticket:      {"event_name": "", "venue": "", "date": "", "time": "", "seat": ""}
- research:    {"topic": "", "key_concepts": [], "source": ""}
- bill:        {"merchant": "", "amount": "", "date": "", "category": ""}
- code_error:  {"error_type": "", "language": "", "error_message": ""}
- social_rec:  {"place_name": "", "location": "", "category": "", "why": ""}
- handwritten: {"subject": "", "key_points": []}
- other:       {"description": ""}

CRITICAL RULES:
- Return ONLY the JSON object. No explanation, no markdown, no backticks.
- If unsure between two types, pick the more specific one and lower the confidence.
- Never return confidence > 0.95 for noisy/partial OCR text.
"""

def classify_intent(redacted_text, source_filename=""):
    """
    Classify the intent of OCR text using LLaMA 3 via Groq.
    Returns a dict with type, confidence, summary, key_data, tags.
    """
    print(f"\n🧠 Classifying intent...")

    # Trim to 3000 chars — keeps us well within free tier token limits
    text_to_send = redacted_text[:3000]
    if len(redacted_text) > 3000:
        print(f"   (Text trimmed from {len(redacted_text)} to 3000 chars for classification)")

    try:
        response = client.chat.completions.create(
            model=MODEL, # Changed from "llama3-8b-8192" to MODEL
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Classify this OCR text:\n\n{text_to_send}"}
            ],
            temperature=0.1,    # Low = deterministic, consistent classification
            max_tokens=500,
        )

        raw_response = response.choices[0].message.content.strip()

        # Clean up in case model adds markdown fences despite instructions
        raw_response = raw_response.replace("```json", "").replace("```", "").strip()

        result = json.loads(raw_response)

        # Validate required fields exist
        for field in ['type', 'confidence', 'summary', 'key_data', 'tags']:
            if field not in result:
                result[field] = 'other' if field == 'type' else (
                    0.5 if field == 'confidence' else (
                    [] if field == 'tags' else (
                    {} if field == 'key_data' else 'Could not extract summary')))

        conf_pct = f"{result['confidence']:.0%}"
        print(f"   ✅ Type: {result['type'].upper()} | Confidence: {conf_pct}")
        print(f"   📝 Summary: {result['summary']}")
        print(f"   🏷️  Tags: {result['tags']}")

        # Safety fallback for very low confidence
        if result['confidence'] < 0.45:
            print(f"   ⚠️  Low confidence — routing to 'other' for safety")
            result['type'] = 'other'

        return result

    except json.JSONDecodeError as e:
        print(f"   ⚠️  LLM returned non-JSON: {e}")
        print(f"   Raw response was: {raw_response[:200]}")
        # Return safe fallback
        return {
            "type": "other",
            "confidence": 0.3,
            "summary": "Classification failed — saved as unstructured note",
            "key_data": {"raw": redacted_text[:500]},
            "tags": ["unclassified"]
        }

print("✅ Classification function loaded. Ready for Cell 7.")

✅ Classification function loaded. Ready for Cell 7.


In [ ]:
# ── CELL 7: Action Router ─────────────────────────────────────────────────────

import csv
import datetime
import json
import re
import os
from pathlib import Path

# ── ROUTER ────────────────────────────────────────────────────────────────────
def route_and_execute(classification, redacted_text, source_path):
    content_type = classification['type']
    print(f"\n⚡ Routing to handler: {content_type.upper()}")

    handlers = {
        'ticket':      lambda: handle_ticket(classification),
        'research':    lambda: handle_research(classification, redacted_text),
        'bill':        lambda: handle_bill(classification),
        'code_error':  lambda: handle_code_error(classification, redacted_text),
        'social_rec':  lambda: handle_social_rec(classification),
        'handwritten': lambda: handle_research(classification, redacted_text),
        'other':       lambda: handle_default(classification, redacted_text, source_path),
    }

    handler = handlers.get(content_type, handlers['other'])
    return handler()


# ── HANDLER: Research / Lecture / Whiteboard ──────────────────────────────────
def handle_research(clf, text):
    date  = datetime.date.today().isoformat()
    kd    = clf.get('key_data', {})
    tags  = clf.get('tags', [])
    topic = tags[0] if tags else 'general'

    safe_topic = re.sub(r'[^\w\-]', '_', topic.lower())
    fname = f"knowledge_base/{date}_{safe_topic}.md"

    tag_str = ' '.join(f'#{t}' for t in tags)
    content = f"""# {clf['summary']}

**Date:** {date}
**Type:** {clf['type']}
**Tags:** {tag_str}
**Confidence:** {clf['confidence']:.0%}

## Summary
{clf['summary']}

## Key Data
{json.dumps(kd, indent=2)}

## Full Extracted Text
{text}
"""
    Path(fname).write_text(content, encoding='utf-8')
    print(f"   📝 Saved: {fname}")
    return fname


# ── HANDLER: Bill / Receipt ───────────────────────────────────────────────────
def handle_bill(clf):
    kd = clf.get('key_data', {})
    row = {
        'date':     datetime.date.today().isoformat(),
        'merchant': kd.get('merchant', 'Unknown'),
        'amount':   kd.get('amount', 'Unknown'),
        'category': kd.get('category', 'other'),
        'summary':  clf['summary'],
    }

    csv_path = 'exports/expenses.csv'
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    print(f"   💸 Expense logged: {row['merchant']} — {row['amount']}")
    return row


# ── HANDLER: Event Ticket → Calendar CSV ─────────────────────────────────────
def handle_ticket(clf):
    kd = clf.get('key_data', {})
    event = {
        'date':       datetime.date.today().isoformat(),
        'event_name': kd.get('event_name', clf['summary']),
        'venue':      kd.get('venue', 'Unknown'),
        'event_date': kd.get('date', 'Unknown'),
        'event_time': kd.get('time', 'Unknown'),
        'seat':       kd.get('seat', 'Unknown'),
    }

    csv_path = 'exports/calendar_events.csv'
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=event.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(event)

    print(f"   📅 Event logged: {event['event_name']} on {event['event_date']}")
    return event


# ── HANDLER: Code Error ───────────────────────────────────────────────────────
def handle_code_error(clf, text):
    kd         = clf.get('key_data', {})
    error_type = re.sub(r'[^\w\-]', '_', kd.get('error_type', 'unknown').lower())
    date       = datetime.date.today().isoformat()
    fname      = f"knowledge_base/{date}_code_error_{error_type}.md"

    so_query = kd.get('error_type', '').replace(' ', '+')
    content = f"""# Code Error: {clf['summary']}

**Date:** {date}
**Error Type:** {kd.get('error_type', 'Unknown')}
**Language:** {kd.get('language', 'Unknown')}

## Error Message
{kd.get('error_message', text[:500])}

## Search Links
- StackOverflow: https://stackoverflow.com/search?q={so_query}
- GitHub: https://github.com/search?q={so_query}

## Full Context
{text}
"""
    Path(fname).write_text(content, encoding='utf-8')
    print(f"   🐛 Code error saved: {fname}")
    return fname


# ── HANDLER: Social Recommendation ───────────────────────────────────────────
def handle_social_rec(clf):
    kd = clf.get('key_data', {})
    row = {
        'date':     datetime.date.today().isoformat(),
        'place':    kd.get('place_name', clf['summary']),
        'location': kd.get('location', 'Unknown'),
        'category': kd.get('category', 'general'),
        'why':      kd.get('why', ''),
        'tags':     str(clf.get('tags', [])),
    }

    csv_path = 'exports/wishlist.csv'
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    print(f"   📍 Saved: {row['place']} ({row['location']})")
    return row


# ── HANDLER: Default / Other ──────────────────────────────────────────────────
def handle_default(clf, text, source_path):
    date  = datetime.date.today().isoformat()
    fname = f"knowledge_base/{date}_capture.md"

    content = f"""# Captured: {clf['summary']}

**Date:** {date}
**Source:** {source_path}
**Tags:** {clf.get('tags', [])}

## Content
{text}
"""
    Path(fname).write_text(content, encoding='utf-8')
    print(f"   📦 Saved as general capture: {fname}")
    return fname


print("✅ All handlers loaded. Ready for Cell 8.")

✅ All handlers loaded. Ready for Cell 8.


In [ ]:
# ── CELL 8: ChromaDB Knowledge Base ──────────────────────────────────────────
# Local vector database. No server, no cloud.
# Persists to Google Drive if you mount it (see comment below).

import chromadb
from sentence_transformers import SentenceTransformer
import uuid

print("Loading embedding model (all-MiniLM-L6-v2, ~22MB)...")
print("This takes 30–60 seconds on first run, then it's cached.")

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedder loaded")

# ── Storage path ──────────────────────────────────────────────────────────────
# Option A: In-session only (resets when Colab restarts)
CHROMA_PATH = "./chroma_store"

# Option B: Persistent via Google Drive (survives restarts)
# from google.colab import drive
# drive.mount('/content/drive')
# CHROMA_PATH = "/content/drive/MyDrive/ClawSight/chroma_store"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
kb = chroma_client.get_or_create_collection(
    name="clawsight_memory",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for semantic search
)
print(f"✅ ChromaDB ready at {CHROMA_PATH}")
print(f"   Entries currently stored: {kb.count()}")

# ── Add to knowledge base ─────────────────────────────────────────────────────
def kb_add(text, metadata):
    """Embed and store a processed entry."""
    entry_id  = str(uuid.uuid4())
    embedding = embedder.encode(text, show_progress_bar=False).tolist()

    kb.add(
        documents=[text],
        embeddings=[embedding],
        metadatas=[metadata],
        ids=[entry_id]
    )
    print(f"   🗄️  Stored in KB — ID: {entry_id[:8]}... | Total entries: {kb.count()}")
    return entry_id

# ── Query in natural language ─────────────────────────────────────────────────
def kb_query(question, n_results=3, filter_type=None):
    """
    Ask a natural language question. Returns semantically similar entries.

    Args:
        question:    Natural language query string
        n_results:   How many results to return (default 3)
        filter_type: Optional — filter by content type
                     e.g. filter_type='research' or 'bill' or 'ticket'
    """
    print(f"\n🔎 Query: \"{question}\"")

    q_embedding = embedder.encode(question, show_progress_bar=False).tolist()

    where_filter = {"type": filter_type} if filter_type else None

    results = kb.query(
        query_embeddings=[q_embedding],
        n_results=min(n_results, max(kb.count(), 1)),
        include=['documents', 'metadatas', 'distances'],
        where=where_filter
    )

    docs      = results['documents'][0]
    metas     = results['metadatas'][0]
    distances = results['distances'][0]

    if not docs:
        print("   No results found.")
        return []

    print(f"   Found {len(docs)} result(s):\n")
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, distances)):
        relevance = max(0, 1 - dist)
        print(f"   [{i+1}] Relevance: {relevance:.0%}")
        print(f"        Type: {meta.get('type', '?')} | Date: {meta.get('date', '?')}")
        print(f"        Summary: {meta.get('summary', '')}")
        print(f"        Preview: {doc[:150]}...")
        print()

    return list(zip(docs, metas, distances))

print("\n✅ Knowledge base ready. Ready for Cell 9 (Full Pipeline).")

Loading embedding model (all-MiniLM-L6-v2, ~22MB)...
This takes 30–60 seconds on first run, then it's cached.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder loaded
✅ ChromaDB ready at ./chroma_store
   Entries currently stored: 0

✅ Knowledge base ready. Ready for Cell 9 (Full Pipeline).


In [ ]:
# ── CELL 9: Full Pipeline ─────────────────────────────────────────────────────
# Processes a single image through all 6 layers.
# Also includes a batch runner for everything in inbox/.

def process_image(image_path):
    """
    Run a single image through the complete ClawSight pipeline.
    Returns the classification result or None on failure.
    """
    sep = "═" * 55
    print(f"\n{sep}")
    print(f"🚀 Processing: {os.path.basename(image_path)}")
    print(sep)

    # ── LAYER 2: OCR ──────────────────────────────────────────────────────────
    try:
        raw_text, confidence = run_ocr(image_path)
    except Exception as e:
        print(f"   ❌ OCR failed: {e}")
        return None

    if not raw_text or len(raw_text.strip()) < 10:
        print("   ❌ OCR extracted no usable text. Check image quality.")
        return None

    # ── LAYER 3: REDACTION (hard gate) ───────────────────────────────────────
    clean_text = safe_redact(raw_text)
    if clean_text is None:
        print("   ❌ Redaction failed — pipeline halted for this image.")
        return None

    # ── LAYER 4: CLASSIFICATION ───────────────────────────────────────────────
    classification = classify_intent(clean_text, source_filename=image_path)

    # ── LAYER 5: ROUTING ──────────────────────────────────────────────────────
    route_result = route_and_execute(classification, clean_text, image_path)

    # ── LAYER 6: STORE IN KNOWLEDGE BASE ──────────────────────────────────────
    kb_add(
        text=clean_text,
        metadata={
            'type':    classification['type'],
            'summary': classification['summary'],
            'tags':    str(classification['tags']),
            'date':    datetime.date.today().isoformat(),
            'source':  os.path.basename(image_path),
            'confidence': str(classification['confidence']),
        }
    )

    # Move processed image out of inbox
    processed_path = f"processed/{os.path.basename(image_path)}"
    os.rename(image_path, processed_path)
    print(f"   📦 Moved to processed/")

    print(f"\n✅ Pipeline complete for: {os.path.basename(image_path)}")
    return classification


def run_batch():
    """Process all images currently sitting in inbox/."""
    images = list(Path('inbox').glob('*'))
    images = [str(f) for f in images if f.suffix.lower() in SUPPORTED_EXTENSIONS]

    if not images:
        print("📭 inbox/ is empty. Upload images using the Files panel → inbox/")
        return

    print(f"📬 Found {len(images)} image(s) in inbox/. Processing...")
    results = []
    for img_path in images:
        result = process_image(img_path)
        if result:
            results.append(result)

    print(f"\n{'═'*55}")
    print(f"✅ Batch complete: {len(results)}/{len(images)} processed successfully")
    return results


# ── RUN: Process everything in inbox/ right now ───────────────────────────────
run_batch()

📭 inbox/ is empty. Upload images using the Files panel → inbox/


In [ ]:
# ── CELL 10: Query — ask anything in natural language ─────────────────────────
# Run this cell whenever you want to search what ClawSight has captured.

print(f"📚 Knowledge base has {kb.count()} entries\n")

# ── Change these queries to whatever you want to find ─────────────────────────
kb_query("circuit diagram analog electronics")
kb_query("restaurant recommendation food")
kb_query("bus train ticket timing")
kb_query("expense receipt total amount")
kb_query("error python code fix")

# ── Filter by type ─────────────────────────────────────────────────────────────
# Uncomment to search only within a specific category:
# kb_query("anything", filter_type="research")
# kb_query("anything", filter_type="bill")
# kb_query("anything", filter_type="ticket")

📚 Knowledge base has 0 entries


🔎 Query: "circuit diagram analog electronics"
   No results found.

🔎 Query: "restaurant recommendation food"
   No results found.

🔎 Query: "bus train ticket timing"
   No results found.

🔎 Query: "expense receipt total amount"
   No results found.

🔎 Query: "error python code fix"
   No results found.


[]

In [ ]:
# ── CELL 11: Synthetic Test — verify pipeline without an image ────────────────
# Creates a fake OCR text and runs it through Layers 3–6.
# Use this to confirm everything is wired correctly before testing real images.

print("🧪 Running synthetic pipeline test...\n")

# Simulated OCR output from a lecture slide photo
FAKE_OCR = """
Neuromorphic Computing: Intel Loihi 2 Architecture
RVCE Technical Seminar — April 2026

Key Concepts:
- Spike-Timing Dependent Plasticity (STDP)
- Event-driven computation (no clock cycles)
- 1,000x lower power than GPU equivalents
- Applications: edge inference, robotics, IoT sensors

Loihi 2 Specs:
- 1 million neurons per chip
- 120 million synapses
- 30mW typical power draw

Contact: speaker@iisc.ac.in  Phone: 9876543210
"""

print("Input text (simulated OCR from lecture slide):")
print("─" * 40)
print(FAKE_OCR[:300] + "...")
print("─" * 40)

# Run through pipeline from Layer 3 onwards
print("\n--- LAYER 3: REDACTION ---")
clean = safe_redact(FAKE_OCR)
print(f"Clean text preview: {clean[:200]}...")

print("\n--- LAYER 4: CLASSIFICATION ---")
clf = classify_intent(clean)

print("\n--- LAYER 5: ROUTING ---")
route_and_execute(clf, clean, "synthetic_test.jpg")

print("\n--- LAYER 6: KNOWLEDGE BASE ---")
kb_add(clean, {
    'type':    clf['type'],
    'summary': clf['summary'],
    'tags':    str(clf['tags']),
    'date':    datetime.date.today().isoformat(),
    'source':  'synthetic_test',
    'confidence': str(clf['confidence']),
})

print("\n--- QUERY TEST ---")
kb_query("neuromorphic chip power consumption")

print("\n✅ Synthetic test complete. If you see a query result above, everything works.")

🧪 Running synthetic pipeline test...

Input text (simulated OCR from lecture slide):
────────────────────────────────────────

Neuromorphic Computing: Intel Loihi 2 Architecture
RVCE Technical Seminar — April 2026

Key Concepts:
- Spike-Timing Dependent Plasticity (STDP)
- Event-driven computation (no clock cycles)
- 1,000x lower power than GPU equivalents
- Applications: edge inference, robotics, IoT sensors

Loihi 2 Spec...
────────────────────────────────────────

--- LAYER 3: REDACTION ---
   🔒 Redacted: email: 1 instance(s), phone_in: 1 instance(s), NER ORG: 'IoT', NER ORG: 'GPU', NER ORG: 'Intel' (+1 more)
Clean text preview: 
[REDACTED_PERSON]: [REDACTED_ORG] Loihi 2 Architecture
RVCE Technical Seminar — April 2026

Key Concepts:
- Spike-Timing Dependent Plasticity (STDP)
- Event-driven computation (no clock cycles)
- 1,0...

--- LAYER 4: CLASSIFICATION ---

🧠 Classifying intent...
   ✅ Type: RESEARCH | Confidence: 92%
   📝 Summary: This image contains a technical seminar presen

Chatbot

In [ ]:
# ── CELL 12: ClawSight Chat Interface ─────────────────────────────────────────
# Upload an image → pipeline runs automatically → chat with your knowledge base
# This is the full demo cell. Run all previous cells first.

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64
from pathlib import Path
import datetime, os, time

# ── Conversation memory (persists across messages in this session) ────────────
conversation_history = []

# ── System prompt for the chat agent ─────────────────────────────────────────
CHAT_SYSTEM_PROMPT = """You are ClawSight, an intelligent personal knowledge assistant.

You have access to the user's personal knowledge base — everything they've ever photographed,
screenshotted, or captured with their phone has been processed and stored.

Your knowledge base contains entries of these types:
- research: lecture slides, whiteboards, academic papers, diagrams
- bill: receipts, invoices, expenses
- ticket: event tickets, bus/train bookings
- code_error: terminal errors, bug logs, stack traces
- social_rec: restaurant/place recommendations from social media
- handwritten: handwritten notes, whiteboard captures
- other: anything else captured

When answering:
- Be conversational and direct — you're a personal assistant, not a search engine
- If you find relevant entries, summarise what you know and cite the date/type
- If nothing relevant exists, say so clearly and suggest what kind of image would help
- You can answer follow-up questions using conversation history
- Keep responses concise — 2-4 sentences unless the user asks for detail

The user's knowledge base contents will be provided with each message as context.
"""

def get_kb_context(question, n=5):
    """Pull the most relevant KB entries for the current question."""
    if kb.count() == 0:
        return "Knowledge base is empty. No images have been processed yet."

    try:
        q_emb = embedder.encode(question, show_progress_bar=False).tolist()
        results = kb.query(
            query_embeddings=[q_emb],
            n_results=min(n, kb.count()),
            include=['documents', 'metadatas', 'distances']
        )

        context_parts = []
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ):
            relevance = max(0, 1 - dist)
            if relevance > 0.2:  # Only include reasonably relevant entries
                context_parts.append(
                    f"[{meta.get('type','?').upper()} | {meta.get('date','?')} | "
                    f"Relevance: {relevance:.0%}]\n"
                    f"Summary: {meta.get('summary','')}\n"
                    f"Content: {doc[:400]}"
                )

        if not context_parts:
            return "No relevant entries found in knowledge base for this query."

        return "KNOWLEDGE BASE CONTEXT:\n\n" + "\n\n---\n\n".join(context_parts)
    except Exception as e:
        return f"Could not query knowledge base: {e}"

def chat_with_kb(user_message):
    """Send a message to ClawSight with KB context. Returns assistant reply."""
    kb_context = get_kb_context(user_message)

    # Build messages: system + KB context + conversation history + new message
    messages = [
        {"role": "system", "content": CHAT_SYSTEM_PROMPT},
        {"role": "system", "content": kb_context},
    ]

    # Add conversation history (last 6 turns to stay within token limits)
    messages.extend(conversation_history[-6:])

    # Add current message
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=600,
    )

    reply = response.choices[0].message.content.strip()

    # Save to conversation history
    conversation_history.append({"role": "user",      "content": user_message})
    conversation_history.append({"role": "assistant", "content": reply})

    return reply

# ── UI Styles ─────────────────────────────────────────────────────────────────
UI_STYLES = """
<style>
  .cs-container {
    font-family: 'Segoe UI', sans-serif;
    max-width: 720px;
    margin: 0 auto;
    background: #0D1B2A;
    border-radius: 12px;
    overflow: hidden;
    border: 1px solid #1E3A5F;
  }
  .cs-header {
    background: linear-gradient(135deg, #0D1B2A, #1B3A5C);
    padding: 16px 20px;
    border-bottom: 2px solid #00B4A0;
    display: flex;
    align-items: center;
    gap: 10px;
  }
  .cs-header-title { color: #fff; font-size: 18px; font-weight: 700; }
  .cs-header-sub   { color: #00B4A0; font-size: 11px; margin-top: 2px; }
  .cs-kb-stats {
    background: #0F2233;
    padding: 8px 20px;
    color: #5A8FA8;
    font-size: 11px;
    border-bottom: 1px solid #1E3A5F;
  }
  .cs-messages {
    height: 380px;
    overflow-y: auto;
    padding: 16px;
    display: flex;
    flex-direction: column;
    gap: 10px;
    background: #0D1B2A;
  }
  .cs-msg-row-user { display:flex; justify-content:flex-end; }
  .cs-msg-row-bot  { display:flex; justify-content:flex-start; gap:8px; align-items:flex-start; }
  .cs-bubble-user {
    background: #00B4A0;
    color: #fff;
    padding: 10px 14px;
    border-radius: 18px 18px 4px 18px;
    max-width: 75%;
    font-size: 13px;
    line-height: 1.5;
  }
  .cs-bubble-bot {
    background: #1B2E42;
    color: #D4E6F1;
    padding: 10px 14px;
    border-radius: 18px 18px 18px 4px;
    max-width: 80%;
    font-size: 13px;
    line-height: 1.6;
    border: 1px solid #1E3A5F;
  }
  .cs-avatar {
    width: 28px; height: 28px;
    background: #00B4A0;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 14px; flex-shrink: 0;
  }
  .cs-timestamp {
    font-size: 10px;
    color: #3A5A6A;
    margin-top: 3px;
    text-align: right;
  }
  .cs-pipeline-log {
    background: #091520;
    border: 1px solid #1E3A5F;
    border-radius: 8px;
    padding: 10px 14px;
    margin: 4px 0;
    font-family: 'Courier New', monospace;
    font-size: 11px;
    color: #4A9E6A;
    max-width: 85%;
  }
  .cs-pipeline-title {
    color: #F5A623;
    font-weight: bold;
    font-size: 11px;
    margin-bottom: 4px;
  }
  .cs-upload-area {
    background: #0F2233;
    border-top: 1px solid #1E3A5F;
    padding: 12px 16px;
  }
  .cs-typing {
    color: #3A8A7A;
    font-size: 12px;
    font-style: italic;
    padding: 4px 8px;
  }
</style>
"""

# ── Build the widget UI ───────────────────────────────────────────────────────
output_area  = widgets.Output()
chat_display = widgets.HTML(value="")
status_bar   = widgets.HTML(value="")

chat_input = widgets.Text(
    placeholder='Ask anything about your captured images... or upload one above',
    layout=widgets.Layout(width='100%', height='38px')
)

send_btn = widgets.Button(
    description='Send',
    button_style='',
    layout=widgets.Layout(width='80px', height='38px'),
    style={'button_color': '#00B4A0', 'font_weight': 'bold'}
)

upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.bmp,.tiff,.webp',
    multiple=False,
    description='📷 Upload Image',
    layout=widgets.Layout(width='160px', height='38px'),
)

clear_btn = widgets.Button(
    description='Clear Chat',
    layout=widgets.Layout(width='100px', height='38px'),
    style={'button_color': '#1B2E42'}
)

# Store chat messages as HTML string
chat_messages_html = []

def get_timestamp():
    return datetime.datetime.now().strftime("%H:%M")

def add_message(role, content, is_pipeline=False):
    """Add a message bubble to the chat display."""
    ts = get_timestamp()

    if is_pipeline:
        # Pipeline log style
        lines = content.replace('\n', '<br>')
        bubble = f"""
        <div class="cs-msg-row-bot">
          <div class="cs-avatar">⚙️</div>
          <div>
            <div class="cs-pipeline-log">
              <div class="cs-pipeline-title">📸 Pipeline Output</div>
              {lines}
            </div>
            <div class="cs-timestamp">{ts}</div>
          </div>
        </div>"""
    elif role == 'user':
        bubble = f"""
        <div class="cs-msg-row-user">
          <div>
            <div class="cs-bubble-user">{content}</div>
            <div class="cs-timestamp">{ts}</div>
          </div>
        </div>"""
    else:
        # Convert newlines to <br> for display
        formatted = content.replace('\n', '<br>')
        bubble = f"""
        <div class="cs-msg-row-bot">
          <div class="cs-avatar">👁️</div>
          <div>
            <div class="cs-bubble-bot">{formatted}</div>
            <div class="cs-timestamp">{ts}</div>
          </div>
        </div>"""

    chat_messages_html.append(bubble)
    refresh_display()

def refresh_display():
    """Rebuild the full chat UI."""
    kb_count = kb.count() if 'kb' in dir() else 0
    messages_html = '\n'.join(chat_messages_html)

    full_html = f"""
    {UI_STYLES}
    <div class="cs-container">
      <div class="cs-header">
        <div>
          <div class="cs-header-title">👁️ ClawSight</div>
          <div class="cs-header-sub">Autonomous Visual Knowledge Agent</div>
        </div>
      </div>
      <div class="cs-kb-stats">
        📚 Knowledge base: {kb_count} entries stored
        &nbsp;·&nbsp;
        💬 Conversation: {len(conversation_history)//2} turns
      </div>
      <div class="cs-messages" id="cs-messages">
        {messages_html}
      </div>
    </div>
    <script>
      // Auto-scroll to bottom
      var el = document.getElementById('cs-messages');
      if(el) el.scrollTop = el.scrollHeight;
    </script>
    """
    chat_display.value = full_html

def show_typing():
    status_bar.value = '<div class="cs-typing">ClawSight is thinking...</div>'

def clear_typing():
    status_bar.value = ''

# ── Event: Send message ───────────────────────────────────────────────────────
def on_send(b=None):
    msg = chat_input.value.strip()
    if not msg:
        return

    chat_input.value = ''
    add_message('user', msg)
    show_typing()

    try:
        reply = chat_with_kb(msg)
        clear_typing()
        add_message('bot', reply)
    except Exception as e:
        clear_typing()
        add_message('bot', f"⚠️ Error: {e}")

# Allow Enter key to send
def on_input_submit(change):
    if change['new']:
        on_send()

chat_input.on_submit(lambda x: on_send())
send_btn.on_click(on_send)

# ── Event: Upload image ───────────────────────────────────────────────────────
def on_upload(change):
    if not upload_btn.value:
        return

    # Get uploaded file
    uploaded_file = list(upload_btn.value.values())[0]
    filename      = list(upload_btn.value.keys())[0]
    file_content  = uploaded_file['content']

    # Save to inbox/
    save_path = f"inbox/{filename}"
    with open(save_path, 'wb') as f:
        f.write(file_content)

    add_message('user', f"📷 Uploaded: <b>{filename}</b>")

    # Capture pipeline output
    import io
    from contextlib import redirect_stdout

    pipeline_output = io.StringIO()
    result = None

    with redirect_stdout(pipeline_output):
        try:
            result = process_image(save_path)
        except Exception as e:
            print(f"Pipeline error: {e}")

    log_text = pipeline_output.getvalue()

    # Show pipeline output as a log bubble
    add_message('bot', log_text, is_pipeline=True)

    # Auto-generate a summary message from ClawSight
    if result:
        content_type = result.get('type', 'unknown')
        summary      = result.get('summary', '')
        confidence   = result.get('confidence', 0)
        tags         = result.get('tags', [])

        auto_msg = (
            f"Got it. I've processed your image and identified it as "
            f"**{content_type}** ({confidence:.0%} confidence).\n\n"
            f"{summary}\n\n"
            f"Tags: {', '.join(tags)}. "
            f"It's saved to your knowledge base — ask me anything about it."
        )
        add_message('bot', auto_msg.replace('**', '<b>').replace('**', '</b>'))
    else:
        add_message('bot',
            "I had trouble processing that image. "
            "Try a clearer photo with better lighting, or a screenshot with readable text."
        )

    # Reset uploader so same file can be re-uploaded if needed
    upload_btn.value.clear()
    upload_btn._counter = 0

upload_btn.observe(on_upload, names='value')

# ── Event: Clear chat ─────────────────────────────────────────────────────────
def on_clear(b):
    global conversation_history
    conversation_history = []
    chat_messages_html.clear()
    add_message('bot',
        "Chat cleared. Your knowledge base is still intact — "
        f"{kb.count()} entries stored. What would you like to know?"
    )

clear_btn.on_click(on_clear)

# ── Layout ────────────────────────────────────────────────────────────────────
top_bar     = widgets.HBox([upload_btn, clear_btn],
                layout=widgets.Layout(gap='8px', padding='4px 0'))
input_row   = widgets.HBox([chat_input, send_btn],
                layout=widgets.Layout(gap='8px'))
full_layout = widgets.VBox([
    chat_display,
    status_bar,
    widgets.HTML('<div style="background:#0F2233;padding:10px 16px;border-top:1px solid #1E3A5F;">'),
    top_bar,
    input_row,
    widgets.HTML('</div>'),
], layout=widgets.Layout(max_width='740px', margin='0 auto'))

# ── Initial greeting ──────────────────────────────────────────────────────────
refresh_display()
add_message('bot',
    f"Hey! I'm ClawSight. I can see and remember everything you photograph.\n\n"
    f"📚 Knowledge base: {kb.count()} entries\n\n"
    f"Upload an image using the button below and I'll process it instantly. "
    f"Then ask me anything — 'what did that lecture slide say?', "
    f"'find my restaurant recommendations', 'show me my expenses this week'."
)

display(full_layout)


📁 New image detected: inbox/busticketex.jpeg
